[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/02_Statistics_and_Probability.ipynb)

# Module 2: Statistics and Probability (25 min)

Statistics turns raw data into understanding and tells us whether a pattern is real or just noise. Every ML model is built on these ideas: loss functions are averages, model outputs are probabilities, and model comparisons are hypothesis tests.

## 2.1 Types of data

The data type decides which statistics and charts make sense.

| Type | Sub-type | Meaning | E-commerce example |
|---|---|---|---|
| Numerical | Continuous | Any value in a range | Order value Rs 1,449.50, time on site 7.3 min |
| Numerical | Discrete | Countable whole numbers | Quantity ordered, number of previous orders |
| Categorical | Nominal | Labels with no order | Payment method, city, product category |
| Categorical | Ordinal | Labels with an order | Rating 1-5, city tier 1/2/3, size S/M/L |


For ML, numerical columns are usually scaled, nominal columns are one-hot encoded, and ordinal columns can be mapped to ordered numbers.

## 2.2 Central tendency, spread and outliers

**Central tendency** describes the "typical" value:

- **Mean** = sum of values / number of values. Sensitive to outliers.
- **Median** = middle value of the sorted data. Robust to outliers.
- **Mode** = most frequent value. Useful for categorical data.

**Spread** describes how far values are from the centre:

- **Range** = max - min.
- **Variance** = average squared distance from the mean; **standard deviation** = square root of variance (same units as the data).
- **Percentiles / quartiles**: Q1 = 25th percentile, Q3 = 75th percentile, **IQR** = Q3 - Q1 covers the middle 50% of data.
- **Outlier rule (IQR)**: values below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR.
- **Z-score** = (x - mean) / std: how many standard deviations a value is from the mean. |z| > 3 is unusual.

> **Real-world example**
>
> A store's daily orders are mostly Rs 450 to Rs 700, but one corporate bulk order is Rs 25,000. The mean jumps to about Rs 2,197 while the median stays at Rs 590. Reporting the mean would give the business a wrong picture of a "typical" order. This is the same as the primer's salary example, where the CEO's salary pulls the mean up.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from scipy import stats

# Real-world example: order values (Rs) from one day at a small online store.
# The last order is a bulk corporate purchase - an outlier.
order_values = np.array([450, 520, 610, 480, 700, 650, 530, 590, 610, 560,
                         620, 610, 480, 540, 25000])

print("Mean   :", round(order_values.mean(), 2))
print("Median :", np.median(order_values))
print("Mode   :", stats.mode(order_values, keepdims=False).mode)

# Spread
print("\nRange             :", order_values.max() - order_values.min())
print("Variance (sample) :", round(order_values.var(ddof=1), 2))
print("Std dev  (sample) :", round(order_values.std(ddof=1), 2))

q1, q3 = np.percentile(order_values, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, outlier fences=({lower}, {upper})")
print("Outliers (IQR rule):", order_values[(order_values < lower) | (order_values > upper)])

# Z-score: how many standard deviations a value is from the mean
z = (order_values - order_values.mean()) / order_values.std(ddof=1)
print("Z-score of the Rs 25,000 order:", round(z[-1], 2))

# Effect of removing the outlier
clean = order_values[order_values <= upper]
print(f"\nWithout outlier -> mean {clean.mean():.2f}, median {np.median(clean)}")

# Pandas gives everything in one line
print("\n", pd.Series(order_values, name="order_value").describe().round(2))

## 2.3 Shape of a distribution and visualisation

- **Skewness** measures asymmetry. Right-skewed (positive) data has a long tail of large values: income, order value, house prices. For right-skewed data, mean > median.
- **Kurtosis** measures how heavy the tails are. High kurtosis means extreme values are more common than in a normal distribution.
- **Histogram**: shows the shape of one numeric variable.
- **Box plot**: shows median, quartiles and outliers at a glance.
- **Scatter plot**: shows the relationship between two numeric variables; the **correlation coefficient** (-1 to +1) measures the strength of a linear relationship.

![Figure 3: The 68-95-99.7 rule of the normal distribution and the parts of a box plot.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d03_normal_boxplot.png)
*Figure 3: The 68-95-99.7 rule of the normal distribution and the parts of a box plot.*


> **Real-world example**
>
> Order values on Indian e-commerce sites are strongly right-skewed. Analysts therefore report median order value, and ML engineers often apply a log transform (`np.log1p`) before modelling. We do exactly this in the final project.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(1)
# Order values are typically right-skewed: many small orders, a few large ones
order_values = rng.lognormal(mean=7, sigma=0.6, size=2000)
# Time on site vs order value (positive relationship + noise)
time_on_site = rng.normal(15, 5, 2000).clip(1)
spend = 200 + 60 * time_on_site + rng.normal(0, 250, 2000)

print("Skewness :", round(stats.skew(order_values), 2), "(> 0 means right-skewed)")
print("Kurtosis :", round(stats.kurtosis(order_values), 2), "(> 0 means heavier tails than normal)")
print("Mean vs median:", round(order_values.mean()), "vs", round(np.median(order_values)))
print("Correlation (time on site vs spend):", round(np.corrcoef(time_on_site, spend)[0, 1], 2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(order_values, bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(order_values.mean(), color="red", label="Mean")
axes[0].axvline(np.median(order_values), color="green", label="Median")
axes[0].set_title("Histogram: order value (right-skewed)")
axes[0].legend()

axes[1].boxplot(order_values)
axes[1].set_title("Box plot: dots beyond whiskers are outliers")

axes[2].scatter(time_on_site, spend, s=5, alpha=0.4)
axes[2].set_xlabel("Minutes on site")
axes[2].set_ylabel("Spend (Rs)")
axes[2].set_title("Scatter: time on site vs spend")
plt.tight_layout()
plt.show()

## 2.4 Probability, conditional probability and Bayes' theorem

Probability is a number between 0 and 1 describing how likely an event is. **Conditional probability** P(A | B) is the probability of A given that B has happened. **Bayes' theorem** updates a prior belief with new evidence:

**P(A | B) = P(B | A) x P(A) / P(B)**

- P(A) is the prior (before evidence), P(B | A) is the likelihood, P(A | B) is the posterior (after evidence).
- When the event is rare, even an accurate test produces many false alarms. This is called the base-rate effect.

> **Real-world example**
>
> A fraud model catches 95% of fraud and wrongly flags only 5% of genuine payments. It sounds excellent, but only 1% of payments are fraud. Bayes' theorem shows that only about 16% of flagged payments are actually fraud. The business must plan manual review capacity accordingly. LLMs also work with conditional probability: they predict P(next word | previous words).

In [ ]:
import numpy as np

# Real-world example: a fraud model flags a payment. How likely is it really fraud?
p_fraud = 0.01                 # prior: 1% of payments are fraudulent
p_flag_given_fraud = 0.95      # sensitivity: model catches 95% of fraud
p_flag_given_genuine = 0.05    # false-alarm rate on genuine payments

# Total probability of a flag
p_flag = p_flag_given_fraud * p_fraud + p_flag_given_genuine * (1 - p_fraud)

# Bayes' theorem: P(fraud | flag) = P(flag | fraud) * P(fraud) / P(flag)
p_fraud_given_flag = p_flag_given_fraud * p_fraud / p_flag
print(f"P(flag)          = {p_flag:.4f}")
print(f"P(fraud | flag)  = {p_fraud_given_flag:.3f}")
print(f"-> only ~{p_fraud_given_flag:.0%} of flagged payments are real fraud")

# Verify with a simulation of 1,000,000 payments
rng = np.random.default_rng(0)
n = 1_000_000
is_fraud = rng.random(n) < p_fraud
flagged = np.where(is_fraud, rng.random(n) < p_flag_given_fraud, rng.random(n) < p_flag_given_genuine)
print(f"Simulated P(fraud | flag) = {is_fraud[flagged].mean():.3f}")

# Conditional probability from a table: P(return | COD) vs P(return | prepaid)
cod_orders, cod_returns = 2000, 520
prepaid_orders, prepaid_returns = 8000, 1200
print(f"\nP(return | COD)     = {cod_returns / cod_orders:.2f}")
print(f"P(return | prepaid) = {prepaid_returns / prepaid_orders:.2f}")

## 2.5 Probability distributions

A probability distribution describes how likely each value of a random variable is. The **expected value** E[X] is the long-run average.

| Distribution | Type | Models | Example |
|---|---|---|---|
| Bernoulli / Binomial | Discrete | Number of successes in n yes/no trials | How many of 20 visitors buy |
| Geometric | Discrete | Trials until the first success | Visitors until the first sale |
| Poisson | Discrete | Count of events in a fixed interval | Orders arriving per minute |
| Uniform | Continuous | All values equally likely | Random A/B group assignment |
| Normal (Gaussian) | Continuous | Symmetric bell curve | Delivery time, measurement errors |
| Exponential | Continuous | Time between events | Seconds between two orders |


The **Central Limit Theorem (CLT)** says that the average of many independent samples is approximately normally distributed, even when the original data is skewed. This is why confidence intervals and t-tests work on real-world data.

> **Real-world example**
>
> Capacity planning for a flash sale: if orders arrive at 4 per minute on average (Poisson), the probability of more than 8 orders in a minute is about 2%. The engineering team uses such numbers to size servers and payment gateways.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Binomial: 20 visitors, each buys with probability 0.10
binom = stats.binom(n=20, p=0.10)
print(f"Binomial  P(exactly 2 buy)   = {binom.pmf(2):.3f}")
print(f"Binomial  P(at least 3 buy)  = {1 - binom.cdf(2):.3f}")

# Geometric: how many visitors until the first purchase?
geom = stats.geom(p=0.10)
print(f"Geometric P(first sale within 5 visitors) = {geom.cdf(5):.3f}")

# Poisson: on average 4 orders arrive per minute
pois = stats.poisson(mu=4)
print(f"Poisson   P(more than 8 orders in a minute) = {1 - pois.cdf(8):.4f}")

# Normal: delivery time ~ N(mean=3 days, sd=1 day)
norm = stats.norm(loc=3, scale=1)
print(f"Normal    P(delivery > 5 days) = {1 - norm.cdf(5):.4f}")
print(f"Normal    95% of deliveries fall between {norm.ppf(0.025):.2f} and {norm.ppf(0.975):.2f} days")

# Exponential: time between orders, average 15 seconds
expo = stats.expon(scale=15)
print(f"Exponential P(next order within 5 s) = {expo.cdf(5):.3f}")

# Central Limit Theorem: averages of a skewed variable become normal
rng = np.random.default_rng(0)
population = rng.exponential(scale=15, size=100_000)          # very skewed
sample_means = [rng.choice(population, 50).mean() for _ in range(3000)]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
k = np.arange(0, 11)
axes[0, 0].bar(k, binom.pmf(k)); axes[0, 0].set_title("Binomial(n=20, p=0.1)")
axes[0, 1].bar(k + 1, geom.pmf(k + 1)); axes[0, 1].set_title("Geometric(p=0.1)")
axes[0, 2].bar(np.arange(0, 13), pois.pmf(np.arange(0, 13))); axes[0, 2].set_title("Poisson(lambda=4)")
x = np.linspace(-1, 7, 200)
axes[1, 0].plot(x, norm.pdf(x)); axes[1, 0].set_title("Normal(3, 1)")
axes[1, 1].hist(population, bins=60, color="orange"); axes[1, 1].set_title("Skewed population")
axes[1, 2].hist(sample_means, bins=40, color="green")
axes[1, 2].set_title("CLT: means of samples (n=50)")
plt.tight_layout()
plt.show()

## 2.6 Statistical inference: confidence intervals and hypothesis tests

We rarely see the whole population, so we estimate from samples.

- **Confidence interval (CI)**: a range that contains the true value with a stated confidence (usually 95%). CI = estimate +/- critical value x standard error.
- **Null hypothesis H0**: "no effect / no difference". **Alternative H1**: "there is an effect".
- **p-value**: the probability of seeing data this extreme if H0 were true. If p < alpha (0.05), reject H0.
- **Type I error**: false alarm (rejecting a true H0). **Type II error**: missing a real effect.

![Figure 4: The five steps of a hypothesis test.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d04_hypothesis.png)
*Figure 4: The five steps of a hypothesis test.*


| Test | Use it when | Example question |
|---|---|---|
| Z-test / two-proportion z-test | Comparing proportions or means with large samples | Is conversion of page B higher than page A? |
| One-sample t-test | Comparing a sample mean with a fixed value | Is average delivery time different from the promised 3 days? |
| Two-sample (Welch) t-test | Comparing means of two groups | Do members spend more than non-members? |
| Chi-square test | Relationship between two categorical variables | Is payment method related to returns? |
| ANOVA | Comparing means of three or more groups | Does average rating differ across App, Web, Desktop? |

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from scipy import stats

rng = np.random.default_rng(10)

# 1) 95% confidence interval for the average order value
sample = rng.normal(1450, 380, 200)           # 200 sampled orders
mean, se = sample.mean(), stats.sem(sample)
ci_low, ci_high = stats.t.interval(0.95, df=len(sample) - 1, loc=mean, scale=se)
print(f"Average order value = {mean:.0f}, 95% CI = ({ci_low:.0f}, {ci_high:.0f})")

# 2) One-sample t-test: is the average delivery time different from the promised 3 days?
delivery = rng.normal(3.3, 1.1, 120)
t, p = stats.ttest_1samp(delivery, popmean=3)
print(f"\nOne-sample t-test: t={t:.2f}, p={p:.4f} ->",
      "Reject H0: delivery differs from 3 days" if p < 0.05 else "Fail to reject H0")

# 3) Two-sample (Welch) t-test: do members spend more than non-members?
members = rng.normal(1600, 400, 150)
non_members = rng.normal(1450, 420, 180)
t, p = stats.ttest_ind(members, non_members, equal_var=False)
print(f"Two-sample t-test: t={t:.2f}, p={p:.4f} ->",
      "Members spend differently" if p < 0.05 else "No evidence of a difference")

# 4) Chi-square test of independence: is payment method related to returns?
table = pd.DataFrame({"Returned": [260, 310, 120], "Kept": [740, 1690, 880]},
                     index=["COD", "UPI", "Card"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"\nChi-square: chi2={chi2:.1f}, dof={dof}, p={p:.2e}")
print("Return rate by payment:\n", (table["Returned"] / table.sum(axis=1)).round(3))

# 5) One-way ANOVA: does the average rating differ across devices?
app = rng.normal(4.2, 0.6, 100)
web = rng.normal(4.0, 0.6, 100)
desktop = rng.normal(4.05, 0.6, 100)
f, p = stats.f_oneway(app, web, desktop)
print(f"\nANOVA: F={f:.2f}, p={p:.4f} ->",
      "At least one device differs" if p < 0.05 else "No significant difference")

## 2.7 Module 2 practical: A/B test of a new checkout page

The product team tested a new one-click checkout page. 12,000 visitors saw the old page (A) and 12,000 saw the new page (B). We test whether the higher conversion of B is real, estimate the uplift with a confidence interval, check basket value, and make a business decision.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# PRACTICAL: A/B test of a new one-click checkout page
# Version A = old page, Version B = new page. Visitors were split randomly.
visitors_a, buyers_a = 12_000, 384      # 3.20% conversion
visitors_b, buyers_b = 12_000, 456      # 3.80% conversion

p_a, p_b = buyers_a / visitors_a, buyers_b / visitors_b
print(f"Conversion A = {p_a:.2%}, Conversion B = {p_b:.2%}, uplift = {(p_b - p_a) / p_a:.1%}")

# Step 1: hypotheses.  H0: p_A = p_B   H1: p_A != p_B   (alpha = 0.05)
# Step 2: two-proportion z-test
p_pool = (buyers_a + buyers_b) / (visitors_a + visitors_b)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / visitors_a + 1 / visitors_b))
z = (p_b - p_a) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"z = {z:.2f}, p-value = {p_value:.4f}")

# Step 3: 95% confidence interval for the difference
se_diff = np.sqrt(p_a * (1 - p_a) / visitors_a + p_b * (1 - p_b) / visitors_b)
low, high = (p_b - p_a) - 1.96 * se_diff, (p_b - p_a) + 1.96 * se_diff
print(f"95% CI for (B - A): {low:.2%} to {high:.2%}")

# Step 4: the same answer with a chi-square test
table = [[buyers_a, visitors_a - buyers_a], [buyers_b, visitors_b - buyers_b]]
chi2, p_chi, _, _ = stats.chi2_contingency(table, correction=False)
print(f"Chi-square p-value = {p_chi:.4f} (matches the z-test)")

# Step 5: did the average basket value change? (Welch t-test on simulated baskets)
rng = np.random.default_rng(3)
basket_a = rng.normal(1480, 400, buyers_a)
basket_b = rng.normal(1495, 410, buyers_b)
t, p_t = stats.ttest_ind(basket_b, basket_a, equal_var=False)
print(f"Basket value t-test p-value = {p_t:.3f}")

# Step 6: business decision
if p_value < 0.05 and low > 0:
    extra = (p_b - p_a) * 100_000 * basket_b.mean()
    print("\nDECISION: roll out version B.")
    print(f"Expected extra revenue per 1 lakh visitors ~ Rs {extra:,.0f}")
else:
    print("\nDECISION: keep version A - the difference is not significant.")

plt.bar(["A (old)", "B (new)"], [p_a * 100, p_b * 100], color=["grey", "seagreen"],
        yerr=[1.96 * np.sqrt(p_a * (1 - p_a) / visitors_a) * 100,
              1.96 * np.sqrt(p_b * (1 - p_b) / visitors_b) * 100], capsize=8)
plt.ylabel("Conversion rate (%)")
plt.title("A/B test result with 95% error bars")
plt.show()

> **Interpretation**
>
> p = 0.011 < 0.05 and the 95% CI for the difference (0.14% to 1.06%) is entirely above zero, so the new page really converts better. Basket value did not change (p = 0.49), so the gain comes from more buyers, not bigger baskets. The same method is used to compare two prompts or two LLM versions in production.